In [ ]:
import pandas as pd
import numpy as np
import os
import glob

def augment_and_rename_imu_data(input_dir, output_dir, noise_std=0.5, scale_factor=1.0):
    """
    對資料夾內的 IMU CSV 數據進行資料增強，並另存為獨立的 other 檔案。
    
    參數 (閾值可供調整):
    - input_dir (str): 原始 CSV 檔案所在的資料夾路徑。
    - output_dir (str): 增強後 CSV 檔案的輸出資料夾路徑。
    - noise_std (float): 高斯雜訊的標準差。值越大，隨機晃動/雜訊越明顯。
    - scale_factor (float): 振幅縮放係數。1.0為不變，1.2表示動作幅度放大20%。
    """
    
    # 確保輸出資料夾存在
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    # 取得所有 CSV 檔案
    csv_files = glob.glob(os.path.join(input_dir, "*.csv"))
    
    if not csv_files:
        print(f"在 {input_dir} 找不到任何 CSV 檔案。")
        return

    # 定義需要增強的感測器基本名稱 (不含 X, Y, Z)
    base_sensors = ['acceleration', 'gyroscope']
    axes = ['_X', '_Y', '_Z']
    
    print(f"找到 {len(csv_files)} 個檔案，準備開始產生 other 數據...")

    for file_path in csv_files:
        file_name = os.path.basename(file_path)
        
        # 處理檔名轉換：將 _notTired 或 _Tired 替換為 _other
        new_file_name = file_name
        if "_notTired_pre1" in new_file_name:
            new_file_name = new_file_name.replace("_notTired_pre1", "_other")
        elif "_notTired_pre2" in new_file_name:
            new_file_name = new_file_name.replace("_notTired_pre2", "_other")
        elif "_notTired_pre" in new_file_name:
            new_file_name = new_file_name.replace("_notTired_pre", "_other")
        elif "_notTired" in new_file_name:
            new_file_name = new_file_name.replace("_notTired", "_other")
            
        elif "_Tired_pre1" in new_file_name:
            new_file_name = new_file_name.replace("_Tired_pre1", "_other")
        elif "_Tired_pre2" in new_file_name:
            new_file_name = new_file_name.replace("_Tired_pre2", "_other")
        elif "_Tired_pre" in new_file_name:
            new_file_name = new_file_name.replace("_Tired_pre", "_other")
        elif "_Tired" in new_file_name:
            new_file_name = new_file_name.replace("_Tired", "_other")
        
            
        output_path = os.path.join(output_dir, new_file_name)
        
        # 讀取原始資料
        try:
            df = pd.read_csv(file_path)
        except Exception as e:
            print(f"讀取 {file_name} 失敗: {e}")
            continue
            
        # 產生所有目標欄位名稱 (包含未矯正與已矯正)
        cols_to_augment = []
        for sensor in base_sensors:
            for axis in axes:
                cols_to_augment.append(f"{sensor}{axis}")             # 未矯正
                cols_to_augment.append(f"{sensor}{axis}_corrected")   # 已矯正
                
        # 直接對原欄位的數值進行覆寫增強 (不新增 _aug 欄位)
        for col in cols_to_augment:
            if col in df.columns:
                # 方法 1: 產生高斯雜訊
                noise = np.random.normal(loc=0.0, scale=noise_std, size=len(df))
                
                # 方法 2: 振幅縮放並加上雜訊，直接覆寫原欄位數值
                df[col] = (df[col] * scale_factor) + noise
                
        # 將結果存入新檔案
        df.to_csv(output_path, index=False)
        print(f"已生成增強檔案: {new_file_name}")

    print("\n所有檔案處理完成！增強後的 Other 數據已獨立生成。")

# ==========================================
# 參數設定區 (請依照你的實驗需求調整這裡的數值)
# ==========================================
if __name__ == "__main__":
    # 設定增強閾值 (核心控制項)
    NOISE_STD = 2.0         # 雜訊強度 (標準差)
    SCALE_FACTOR = 1.2      # 動作放大倍率 (1.0 代表不放大，1.2代表放大20%)
    
    # 設定資料夾路徑 (可以使用相對路徑或絕對路徑)
    INPUT_FOLDER = "./data"      # 放置你原始 CSV (notTired/Tired) 的資料夾
    OUTPUT_FOLDER = "./other"     # 產生出的 other CSV 輸出的資料夾

    # 執行增強程式
    augment_and_rename_imu_data(
        input_dir=INPUT_FOLDER, 
        output_dir=OUTPUT_FOLDER, 
        noise_std=NOISE_STD, 
        scale_factor=SCALE_FACTOR
    )

    INPUT_FOLDER = "./test_data_6"      # 放置你原始 CSV (notTired/Tired) 的資料夾
    OUTPUT_FOLDER = "./test_data_6"     # 產生出的 other CSV 輸出的資料夾

    augment_and_rename_imu_data(
        input_dir=INPUT_FOLDER, 
        output_dir=OUTPUT_FOLDER, 
        noise_std=NOISE_STD, 
        scale_factor=SCALE_FACTOR
    )

In [ ]:
# 直接取代數據，用於增強原數據

import pandas as pd
import numpy as np
import os
import glob

def augment_imu_data(input_dir, output_dir, noise_std=0.5, scale_factor=1.0):
    """
    對資料夾內的 IMU CSV 數據進行資料增強 (直接取代舊欄位)。
    
    參數 (閾值可供調整):
    - input_dir (str): 原始 CSV 檔案所在的資料夾路徑。
    - output_dir (str): 增強後 CSV 檔案的輸出資料夾路徑。
    - noise_std (float): 高斯雜訊的標準差。值越大，隨機晃動/雜訊越明顯 (適合做邊界測試)。
    - scale_factor (float): 振幅縮放係數。1.0為不變，1.2表示動作幅度放大20% (適合做誇張化動作)。
    """
    
    # 確保輸出資料夾存在
    if not os.path.exists(output_dir):
        os.makedirs(output_dir)
        
    # 取得所有 CSV 檔案
    csv_files = glob.glob(os.path.join(input_dir, "*.csv"))
    
    if not csv_files:
        print(f"在 {input_dir} 找不到任何 CSV 檔案。")
        return

    # 定義需要增強的感測器基本名稱 (不含 X, Y, Z)
    base_sensors = ['acceleration', 'gyroscope']
    axes = ['_X', '_Y', '_Z']
    
    print(f"找到 {len(csv_files)} 個檔案，準備開始處理...")

    for file_path in csv_files:
        file_name = os.path.basename(file_path)
        output_path = os.path.join(output_dir, f"{file_name}")
        
        # 讀取原始資料
        try:
            df = pd.read_csv(file_path)
        except Exception as e:
            print(f"讀取 {file_name} 失敗: {e}")
            continue
            
        # 產生所有目標欄位名稱 (包含未矯正與已矯正)
        cols_to_augment = []
        for sensor in base_sensors:
            for axis in axes:
                cols_to_augment.append(f"{sensor}{axis}")             # 未矯正
                cols_to_augment.append(f"{sensor}{axis}_corrected")   # 已矯正
                
        # 紀錄實際修改的欄位數量
        modified_count = 0
        
        # 針對選定欄位進行增強
        for col in cols_to_augment:
            if col in df.columns:
                # 方法 1: 加上高斯雜訊 (模擬不穩定或隨機的晃動)
                noise = np.random.normal(loc=0.0, scale=noise_std, size=len(df))
                
                # 方法 2: 振幅縮放 (模擬動作幅度變大或變小)
                # 將原數值乘上 scale_factor 後加上 noise，並直接覆寫回原本的欄位
                df[col] = (df[col] * scale_factor) + noise
                
                modified_count += 1
                
        # 將結果存入新檔案
        df.to_csv(output_path, index=False)
        print(f"已處理並儲存: {output_path} (修改了 {modified_count} 個原始欄位)")

    print("\n所有檔案處理完成！")

# ==========================================
# 參數設定區 (請依照你的實驗需求調整這裡的數值)
# ==========================================
if __name__ == "__main__":
    
    # 1. 設定資料夾路徑 (可以使用相對路徑或絕對路徑)
    INPUT_FOLDER = "./data"      # 放置你原始 CSV 的資料夾
    OUTPUT_FOLDER = "./aug_data" # 處理後輸出的資料夾

    # 2. 設定增強閾值 (核心控制項)
    # 微小增強 (訓練集擴充)：NOISE_STD = 0.2, SCALE_FACTOR = 1.05
    # 誇張增強 (製造第三類別)：NOISE_STD = 1.0, SCALE_FACTOR = 1.5
    NOISE_STD = 0.3         # 雜訊強度 (標準差)
    SCALE_FACTOR = 1.1      # 動作放大倍率 (1.0 代表不放大，1.2代表放大20%)

    # 執行增強程式
    augment_imu_data(
        input_dir=INPUT_FOLDER, 
        output_dir=OUTPUT_FOLDER, 
        noise_std=NOISE_STD, 
        scale_factor=SCALE_FACTOR
    )